# Stage1 分监督域输出头

假设：主ETF与辅助个股合成2x的标签不同，可能需要不同输出映射；保留共享编码器和辅助梯度。B为原共享头匹配重训，C只克隆并分开辅助tail/alpha头。并不预设负迁移已经成立。

冻结设计见[plan.json](plan.json)，原始方案及独立评审变更见[plan_before_review.json](plan_before_review.json)。2023 cutoff共6次拟合，seed71/172/273全部保留、固定300轮；2020辅助0条不跑。

In [1]:
import os, sys, json
from pathlib import Path
root = Path.cwd()
while not (root / "AGENTS.md").exists():
    if root.parent == root:
        raise RuntimeError("workspace root not found")
    root = root.parent
sys.path.insert(0, str(root))
os.environ["MLFLOW_ALLOW_FILE_STORE"] = "true"
os.environ["MLFLOW_DISABLE_AGENT_HINT"] = "1"
os.environ["MLFLOW_ENABLE_ARTIFACTS_PROGRESS_BAR"] = "false"
import qlib
from qlib.data import D
from qlib.workflow import R
from src.workflow.stage1_domain_heads import prepare, run
card = root / "research/stage1_domain_heads_20261018"
plan = json.loads((card / "plan.json").read_text())
qlib.init(provider_uri=str(root / plan["provider_init"]), region="us", kernels=1)


Gym has been unmaintained since 2022 and does not support NumPy 2.0 amongst other critical functionality.
Please upgrade to Gymnasium, the maintained drop-in replacement of Gym, or contact the authors of your software and request that they upgrade.
Users of this version of Gym should be able to simply replace 'import gym' with 'import gymnasium as gym' in the vast majority of cases.
See the migration guide at https://gymnasium.farama.org/introduction/migration_guide/ for additional information.


[47950:MainThread](2026-10-18 11:35:22,494) INFO - qlib.Initialization - [config.py:452] - default_conf: client.


[47950:MainThread](2026-10-18 11:35:22,495) INFO - qlib.Initialization - [__init__.py:75] - qlib successfully initialized based on client settings.


[47950:MainThread](2026-10-18 11:35:22,496) INFO - qlib.Initialization - [__init__.py:77] - data_path={'__DEFAULT_FREQ': PosixPath('/Users/rziiiii/dev/workspace/data/qlib/minute_information_pit_allocator_20261012/core_1min')}


## 冻结来源与成熟样本预检

只读哈希绑定的Qlib Recorder，经DatasetH.prepare恢复原输入。主3014行、辅助20741行，39主输入与16共享输入；不读取当前脏provider替换来源。A为现役f696+b472在同源09:45已知→09:47成交→10:32恢复研究合同下重放，不是09:31/10:31运行记录。A/B/C独立费用状态、原路由和恢复，无T/COMMON/ALL3。

In [2]:
prepared = prepare(root=root, card=card)
prepared["receipt"]


[47950:MainThread](2026-10-18 11:35:22,508) INFO - qlib.Initialization - [config.py:452] - default_conf: client.


[47950:MainThread](2026-10-18 11:35:22,509) INFO - qlib.Initialization - [__init__.py:75] - qlib successfully initialized based on client settings.


[47950:MainThread](2026-10-18 11:35:22,509) INFO - qlib.Initialization - [__init__.py:77] - data_path={'__DEFAULT_FREQ': PosixPath('/Users/rziiiii/dev/workspace/data/qlib/minute_information_pit_allocator_20261012/core_1min')}


[47950:MainThread](2026-10-18 11:35:25,612) INFO - qlib.timer - [log.py:127] - Time cost: 0.001s | Loading data Done


[47950:MainThread](2026-10-18 11:35:25,613) INFO - qlib.timer - [log.py:127] - Time cost: 0.000s | fit & process data Done


[47950:MainThread](2026-10-18 11:35:25,613) INFO - qlib.timer - [log.py:127] - Time cost: 0.002s | Init data Done


[47950:MainThread](2026-10-18 11:35:25,623) INFO - qlib.timer - [log.py:127] - Time cost: 0.000s | Loading data Done


[47950:MainThread](2026-10-18 11:35:25,623) INFO - qlib.timer - [log.py:127] - Time cost: 0.000s | fit & process data Done


[47950:MainThread](2026-10-18 11:35:25,623) INFO - qlib.timer - [log.py:127] - Time cost: 0.001s | Init data Done


{'plan_sha256': '96dc55af2318e26162e7ff8f74c4fdf24bcd15c4d9a3c4e1e2ca18357e3e17c7',
 'primary_rows': 3014,
 'auxiliary_rows': 20741,
 'primary_last_mature': '2022-12-30 09:31:00-05:00',
 'aux_first_signal': '2019-11-25 00:00:00',
 'aux_last_signal': '2022-12-22 00:00:00',
 'aux_observations_end_max': '2022-12-30 00:00:00',
 'source_hashes_checked': True,
 'source_api': ['Recorder.load_object', 'DatasetH.prepare'],
 'main_inputs': 39,
 'common_inputs': 16,
 'account_sessions': 944,
 'early_fields_exact_source_equality': ['early_ret_soxl',
  'early_ret_soxx',
  'early_range_soxl',
  'early_range_soxx',
  'early_from_low_soxl',
  'early_from_low_soxx',
  'early_volume_soxl',
  'early_volume_soxx',
  'early_bar_count_soxl',
  'early_bar_count_soxx'],
 'reused_reference_accounts': 9,
 'active_bundle_and_classifier_hashes_verified': True,
 'initialization': [{'seed': 71,
   'original_parameters': 973,
   'candidate_parameters': 991,
   'original_tensors_identical': True,
   'initial_primary_

## 固定训练、完整账户与分析

正式执行还要求独立design_review与owner execution_approval匹配计划及最终代码hash。分类成绩不决定是否运行原定完整账户。主/辅助初始输出相同，新增18参数以及全局梯度裁剪变化属于整体干预。原价格、公司行动与已完成SOXX/9097/27baa九个原生账户严格同口径后复用；新增A/B/C九账户，0/2/5bp分别结算。

In [3]:
execute = os.environ.get("STAGE1_DOMAIN_HEADS_EXECUTE") == "1"
if execute:
    result = run(root=root, card=card, prepared=prepared)
else:
    result = json.loads((card / "summary.json").read_text())
result


{'experiment': 'stage1_domain_heads_20261018',
 'decision': 'implementation_pending_independent_design_review',
 'objective_achieved': False,
 'new_fits': 0,
 'new_native_accounts': 0,
 'artifact_policy': {'policy': 'slim_card_recorder_first',
  'workflow_config': 'workflow.yaml',
  'recorder_only_artifacts': ['inputs/*',
   'models/*',
   'predictions/*',
   'paths/*',
   'native/*',
   'analysis/rolling/*']}}

In [4]:
from IPython.display import Markdown, display
display(Markdown((card / "conclusion.md").read_text()))


# 分监督域输出头：实施准备

已登记仅拆分辅助tail/alpha输出头的配对实验。冻结Recorder预检主3014行、辅助20741行、39主输入与16共享输入。2020切分辅助为零，不执行。当前0拟合、0新增账户；等待独立设计评估与owner明确开跑。

全部历史已经复用，没有独立前向资格；A是现役冻结模型在同源研究时钟下重放，非实盘09:31/10:31复现。不晋升，生产未改，稳定超额目标尚未达到。
